# Lean-RGC Automation Stack Quickstart

This notebook runs the Lean-RGC scaffold. It works in two modes:

1. **Dry run**: no Lean installation needed, validates schemas and pipeline.
2. **Lean mode**: if Lean/Lake + Mathlib are available, runs real file-based micro-audits.

The pipeline implements: MicroAudit → DefectExtractor → Response data → Quotient components → Carrier proposals.

In [ ]:
from pathlib import Path
import os, json, subprocess, sys, shutil
ROOT = Path('/content/lean_rgc_runs')
ROOT.mkdir(parents=True, exist_ok=True)
print(ROOT)

## Upload / unzip package

If you downloaded `lean_rgc_automation_stack.zip`, upload it in Colab and set `ZIP_PATH`.

In [ ]:
ZIP_PATH = '/content/lean_rgc_automation_stack.zip'  # change if needed
if Path(ZIP_PATH).exists():
    !rm -rf /content/lean_rgc_stack
    !unzip -q {ZIP_PATH} -d /content/lean_rgc_unzip
    # archive contains lean_rgc_stack/
    !pip install -q -e /content/lean_rgc_unzip/lean_rgc_stack
else:
    print('ZIP not found. If running from repository, pip install -e the package path manually.')

## Dry-run audit

In [ ]:
PKG = Path('/content/lean_rgc_unzip/lean_rgc_stack')
if not PKG.exists():
    PKG = Path.cwd()
TASKS = PKG / 'examples' / 'minimal_theorems.jsonl'
ACTIONS = PKG / 'examples' / 'tactic_templates.jsonl'
OUT = ROOT / 'dry_audit'
!lean-rgc audit --tasks {TASKS} --actions {ACTIONS} --out {OUT} --dry-run --max-actions 8

In [ ]:
!lean-rgc quotient --responses {OUT/'responses.jsonl'} --out {OUT/'response_components.jsonl'} --tolerance 0.25
!head -n 5 {OUT/'response_components.jsonl'}

## Optional: real Lean audit

Run this only if `/content/lean_project` is a Lake project with Mathlib available.

In [ ]:
LEAN_WORKDIR = '/content/lean_project'
if Path(LEAN_WORKDIR).exists() and shutil.which('lake'):
    OUT2 = ROOT / 'lean_audit'
    !lean-rgc audit --tasks {TASKS} --actions {ACTIONS} --out {OUT2} --lean-cmd 'lake env lean' --workdir {LEAN_WORKDIR} --timeout-s 20 --max-actions 8
else:
    print('No Lean/Lake project found; skipped real Lean audit.')

## Carrier generator from responses

In [ ]:
# Build a small state-defect-like file from response records for demonstration.
import json
rows=[]
for line in open(OUT/'responses.jsonl'):
    r=json.loads(line)
    d=r['defect_before']
    d['state_id']=r['state_id']
    rows.append(d)
DEFS=OUT/'defects_for_carrier.jsonl'
with open(DEFS,'w') as f:
    for r in rows:
        f.write(json.dumps(r)+'\n')
!lean-rgc carrier-generate --defects {DEFS} --out {OUT/'carrier_generated_contexts.jsonl'}
!head -n 5 {OUT/'carrier_generated_contexts.jsonl'}

## Inspect summary

In [ ]:
import pandas as pd, json
responses = [json.loads(l) for l in open(OUT/'responses.jsonl')]
df = pd.DataFrame([{
    'state_id': r['state_id'],
    'action_id': r['action_id'],
    'status': r['audit_status'],
    'response_norm': sum(x*x for x in r['response_flat'])**0.5,
    'carrier_delta_sum': sum(r.get('carrier_delta',{}).values())
} for r in responses])
display(df.head(20))
display(df.groupby('status').agg(n=('status','size'), mean_response_norm=('response_norm','mean')))

## Next steps

- Replace dry run with a real Lake project.
- Add theorem/task JSONL from a selected Mathlib domain.
- Expand candidate generation with premise retrieval and domain-specific tactic templates.
- Train a response learner on `responses.jsonl`.
- Add trajectory logging for Gamma audit.